In [1]:
import pandas as pd

from data_preprocessingagent import DataPreprocessingAgent
from health_monitoring_agent import HealthMonitoringAgent
from fault_detection_agent import FaultDetectionAgent
from risk_analysis_agent import RiskAnalysisAgent

In [2]:
pre_agent=DataPreprocessingAgent()
train,test,rul=pre_agent.preprocess(
    "train_FD003.txt",
    "test_FD003.txt",
    "RUL_FD003.txt"
)
health_agent=HealthMonitoringAgent()
health_df=health_agent.monitor(train)
fault_agent=FaultDetectionAgent()
fault_df=fault_agent.detect(health_df)
risk_agent=RiskAnalysisAgent()
risk_df=risk_agent.analyze(fault_df)

Shape: (24720, 26)
Missing: 0
Duplicates: 0


In [3]:
explain_df=risk_df.copy()

### get sensor columns

In [4]:
sensor_cols=[
    col
    for col in explain_df.columns
    if col.startswith("sensor")
    and "_trend" not in col
    and "_anomaly" not in col
]

### calculate baseline sensor values

In [5]:
baseline=(
    explain_df
    .groupby("engine_id")
    .first()[sensor_cols]
)

### find most influencial sensor

In [6]:
key_factors=[]
for _, row in explain_df.iterrows():
    engine=row["engine_id"]
    base=baseline.loc[engine]
    current=row[sensor_cols]
    deviation=abs(current-base)
    important_sensor=deviation.idxmax()
    key_factors.append(important_sensor)
explain_df["Key_Factor"]=key_factors

### generate explanations

In [8]:
def generate_explanation(row):
    return (
        f"Engine classified as "
        f"{row['Health_Status']} "
        f"with {row['Risk_Level']} risk. "
        f"Primary degradation observed in "
        f"{row['Key_Factor']}."
    )

In [9]:
explain_df["Explanation"]=(
    explain_df.apply(
        generate_explanation,
        axis=1
    )
)

### results

In [10]:
explain_df[
    [
        "engine_id",
        "Health_Status",
        "Risk_Level",
        "Key_Factor",
        "Explanation"
    ]
].head(20)

,engine_id,Health_Status,Risk_Level,Key_Factor,Explanation
0,1,Healthy,Low,sensor2,Engine classified as Healthy with Low risk. Pr...
1,1,Healthy,Low,sensor17,Engine classified as Healthy with Low risk. Pr...
2,1,Healthy,Low,sensor20,Engine classified as Healthy with Low risk. Pr...
3,1,Healthy,Low,sensor2,Engine classified as Healthy with Low risk. Pr...
4,1,Healthy,Low,sensor2,Engine classified as Healthy with Low risk. Pr...
5,1,Healthy,Low,sensor17,Engine classified as Healthy with Low risk. Pr...
6,1,Healthy,Low,sensor20,Engine classified as Healthy with Low risk. Pr...
7,1,Healthy,Low,sensor3,Engine classified as Healthy with Low risk. Pr...
8,1,Healthy,Low,sensor2,Engine classified as Healthy with Low risk. Pr...
9,1,Healthy,Low,sensor21,Engine classified as Healthy with Low risk. Pr...


In [11]:
explain_df.to_csv(
    "explainability_output.csv",
    index=False
)
print("Explainability Output Saved")

Explainability Output Saved
